In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# used Pima Indians Diabetes dataset 

In [3]:
possible_paths = [
    Path("../datasets/diabetes.csv"),
    Path("datasets/diabetes.csv"),
    Path("diabetes.csv")
]

csv_path = next((p for p in possible_paths if p.exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "Could not find diabetes.csv. Place it in the project's datasets folder "
        "or update csv_path to the correct location."
    )

df = pd.read_csv(csv_path)

print("Dataset path:", csv_path.resolve())
print("Dataset shape:", df.shape)
display(df.head())

Dataset path: E:\Multi_Disease_Prediction_System\datasets\diabetes.csv
Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["Outcome"].value_counts())

print("\nTarget meaning: 0 = No Diabetes, 1 = Diabetes")

Columns:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Missing values per column:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Target meaning: 0 = No Diabetes, 1 = Diabetes


In [5]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

feature_names = X.columns.tolist()

print("Feature columns in training order:")
print(feature_names)
print("\nX shape:", X.shape)
print("y shape:", y.shape)

Feature columns in training order:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

X shape: (768, 8)
y shape: (768,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (614, 8)
X_test : (154, 8)
y_train: (614,)
y_test : (154,)


In [7]:
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

log_accuracy = accuracy_score(y_test, log_pred)
print(f"Logistic Regression Accuracy: {log_accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, log_pred))
print("\nClassification Report:")
print(classification_report(y_test, log_pred, zero_division=0))

Logistic Regression Accuracy: 70.78%

Confusion Matrix:
[[79 21]
 [24 30]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.79      0.78       100
           1       0.59      0.56      0.57        54

    accuracy                           0.71       154
   macro avg       0.68      0.67      0.67       154
weighted avg       0.70      0.71      0.71       154



In [8]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, zero_division=0))

Random Forest Accuracy: 86.36%

Confusion Matrix:
[[90 10]
 [11 43]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.90      0.90       100
           1       0.81      0.80      0.80        54

    accuracy                           0.86       154
   macro avg       0.85      0.85      0.85       154
weighted avg       0.86      0.86      0.86       154



In [9]:
log_cv = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train, y_train, cv=5, scoring="accuracy"
)
rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train, cv=5, scoring="accuracy"
)

print(f"Logistic Regression CV scores: {np.round(log_cv, 4)}")
print(f"Logistic Regression CV mean:   {log_cv.mean() * 100:.2f}%")
print()
print(f"Random Forest CV scores:       {np.round(rf_cv, 4)}")
print(f"Random Forest CV mean:         {rf_cv.mean() * 100:.2f}%")

Logistic Regression CV scores: [0.7967 0.8049 0.7724 0.7886 0.8033]
Logistic Regression CV mean:   79.32%

Random Forest CV scores:       [0.8618 0.878  0.878  0.8943 0.8689]
Random Forest CV mean:         87.62%


In [10]:
log_cv = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train, y_train, cv=5, scoring="accuracy"
)
rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train, cv=5, scoring="accuracy"
)

print(f"Logistic Regression CV scores: {np.round(log_cv, 4)}")
print(f"Logistic Regression CV mean:   {log_cv.mean() * 100:.2f}%")
print()
print(f"Random Forest CV scores:       {np.round(rf_cv, 4)}")
print(f"Random Forest CV mean:         {rf_cv.mean() * 100:.2f}%")

Logistic Regression CV scores: [0.7967 0.8049 0.7724 0.7886 0.8033]
Logistic Regression CV mean:   79.32%

Random Forest CV scores:       [0.8618 0.878  0.878  0.8943 0.8689]
Random Forest CV mean:         87.62%


In [11]:
if rf_cv.mean() >= log_cv.mean():
    best_model = rf_model
    best_model_name = "Random Forest"
    best_test_accuracy = rf_accuracy
else:
    best_model = log_model
    best_model_name = "Logistic Regression"
    best_test_accuracy = log_accuracy

print("Selected model:", best_model_name)
print(f"Selected model test accuracy: {best_test_accuracy * 100:.2f}%")
print("Selected model CV mean:", f"{max(log_cv.mean(), rf_cv.mean()) * 100:.2f}%")

Selected model: Random Forest
Selected model test accuracy: 86.36%
Selected model CV mean: 87.62%


In [12]:
model_dir = Path("../models")
if not model_dir.exists():
    model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "diabetes_model.pkl"
joblib.dump(best_model, model_path)

print("Saved model:", model_path.resolve())
print("Model type:", type(best_model).__name__)
print("Feature order expected by the model:")
print(feature_names)

Saved model: E:\Multi_Disease_Prediction_System\models\diabetes_model.pkl
Model type: RandomForestClassifier
Feature order expected by the model:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


In [13]:
sample = pd.DataFrame([{
    "Pregnancies": 1,
    "Glucose": 85,
    "BloodPressure": 66,
    "SkinThickness": 29,
    "Insulin": 14,
    "BMI": 26.6,
    "DiabetesPedigreeFunction": 0.351,
    "Age": 31
}])[feature_names]

sample_prediction = best_model.predict(sample)[0]
print("Predicted class:", int(sample_prediction))
print("Interpretation:", "Diabetes" if sample_prediction == 1 else "No Diabetes")

Predicted class: 0
Interpretation: No Diabetes


In [14]:
from collections import Counter
import numpy as np

print("Logistic Regression predictions:")
print(Counter(log_model.predict(X_test)))

print("\nRandom Forest predictions:")
print(Counter(rf_model.predict(X_test)))

print("\nActual test labels:")
print(Counter(y_test))

print("\nSelected model:", best_model_name)
print("Selected model classes:", best_model.classes_)

Logistic Regression predictions:
Counter({np.int64(0): 103, np.int64(1): 51})

Random Forest predictions:
Counter({np.int64(0): 101, np.int64(1): 53})

Actual test labels:
Counter({0: 100, 1: 54})

Selected model: Random Forest
Selected model classes: [0 1]


In [15]:
# v1

In [16]:
import joblib
import pandas as pd
from pathlib import Path

# Load the saved model
model_path = Path("../models/diabetes_model.pkl")
saved_model = joblib.load(model_path)

print("Saved model type:", type(saved_model).__name__)

# Check the model's expected feature order
if hasattr(saved_model, "feature_names_in_"):
    feature_names = list(saved_model.feature_names_in_)
    print("\nExpected feature order:")
    print(feature_names)
else:
    feature_names = list(X_train.columns) if hasattr(X_train, "columns") else list(df.drop(columns=["Outcome"]).columns)
    print("\nFeature names obtained from the training data:")
    print(feature_names)

# Two illustrative test inputs
test_data = pd.DataFrame(
    [
        [1, 85, 66, 29, 14, 26.6, 0.351, 31],
        [6, 168, 74, 35, 100, 33.6, 0.627, 50]
    ],
    columns=feature_names
)

print("\nPredictions from the saved model:")
print(saved_model.predict(test_data))

print("\nPredicted labels: 0 = No Diabetes, 1 = Diabetes")


Saved model type: RandomForestClassifier

Expected feature order:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Predictions from the saved model:
[0 0]

Predicted labels: 0 = No Diabetes, 1 = Diabetes


In [17]:
# v2 

In [19]:
import numpy as np
import joblib
from pathlib import Path

saved_model = joblib.load(Path("../models/diabetes_model.pkl"))

# Compare predictions on the same test data
notebook_predictions = rf_model.predict(X_test)
saved_predictions = saved_model.predict(X_test)

print("Notebook model predictions:")
print(np.unique(notebook_predictions, return_counts=True))

print("\nSaved model predictions:")
print(np.unique(saved_predictions, return_counts=True))

print("\nAre all predictions identical?")
print(np.array_equal(notebook_predictions, saved_predictions))

print("\nNotebook model accuracy:")
print(rf_model.score(X_test, y_test))

print("\nSaved model accuracy:")
print(saved_model.score(X_test, y_test))


Notebook model predictions:
(array([0, 1]), array([101,  53]))

Saved model predictions:
(array([0, 1]), array([101,  53]))

Are all predictions identical?
True

Notebook model accuracy:
0.8636363636363636

Saved model accuracy:
0.8636363636363636


In [20]:
# v3
import pandas as pd
import joblib
from pathlib import Path

saved_model = joblib.load(Path("../models/diabetes_model.pkl"))

# Find one test sample predicted as Diabetes and one as No Diabetes
test_predictions = saved_model.predict(X_test)

positive_index = next(i for i, p in enumerate(test_predictions) if p == 1)
negative_index = next(i for i, p in enumerate(test_predictions) if p == 0)

for label, index in [
    ("PREDICTED DIABETES", positive_index),
    ("PREDICTED NO DIABETES", negative_index)
]:
    sample = X_test.iloc[[index]]
    prediction = saved_model.predict(sample)[0]

    print(f"\n--- {label} ---")
    print("Input values:")
    print(sample.iloc[0].to_dict())
    print("Actual dataset label:", y_test.iloc[index] if hasattr(y_test, "iloc") else y_test[index])
    print("Saved model prediction:", prediction)



--- PREDICTED DIABETES ---
Input values:
{'Pregnancies': 7.0, 'Glucose': 114.0, 'BloodPressure': 64.0, 'SkinThickness': 32.0, 'Insulin': 169.5, 'BMI': 27.4, 'DiabetesPedigreeFunction': 0.732, 'Age': 34.0}
Actual dataset label: 1
Saved model prediction: 1

--- PREDICTED NO DIABETES ---
Input values:
{'Pregnancies': 7.0, 'Glucose': 159.0, 'BloodPressure': 64.0, 'SkinThickness': 27.0, 'Insulin': 102.5, 'BMI': 27.4, 'DiabetesPedigreeFunction': 0.294, 'Age': 40.0}
Actual dataset label: 0
Saved model prediction: 0
